In [31]:
import sys
from pathlib import Path
from dotenv import load_dotenv

%load_ext autoreload
%autoreload 2


from pathlib import Path

def find_repo_root_by_subfolders(start_path: Path, required_subfolders=("src", "notebooks"), max_up=5) -> Path:
    """
    Walks up from start_path up to max_up levels to find a folder
    containing all required_subfolders.
    Returns the Path to the repo root or raises FileNotFoundError.
    """
    current = start_path.resolve()
    for _ in range(max_up):
        if all((current / subfolder).exists() for subfolder in required_subfolders):
            return current
        current = current.parent
    raise FileNotFoundError(
        f"Could not find repo root containing {required_subfolders} "
        f"within {max_up} levels up from {start_path}"
    )

# Usage example in your notebook/script:
cwd = Path().resolve()
try:
    repo_root = find_repo_root_by_subfolders(cwd)
except FileNotFoundError:
    print("Repo root with /src and /notebooks not found.")
    # Optionally fallback to something else or raise error

print(f"Repo root found at: {repo_root}")
src_path = repo_root / "src"
notebooks_path = repo_root / "notebooks"

# Add src to sys.path for imports
import sys
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Load env, etc.
from dotenv import load_dotenv
load_dotenv(dotenv_path=repo_root / ".env")

print(f"/src added to sys.path: {src_path}")
print(f"Notebook path: {notebooks_path}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Repo root found at: C:\Data\League Work\league-logger\league-logger
/src added to sys.path: C:\Data\League Work\league-logger\league-logger\src
Notebook path: C:\Data\League Work\league-logger\league-logger\notebooks


In [32]:
import os

# Load .env
env_path = repo_root / ".env"
load_dotenv(dotenv_path=env_path)

RIOT_API_KEY = os.getenv("RIOT_API_KEY")
REGION_ROUTING = os.getenv("REGION_ROUTING")

print("RIOT_API_KEY:", os.getenv("RIOT_API_KEY"))

if not RIOT_API_KEY:
    raise ValueError("RIOT_API_KEY not found in .env")

# Import the sync function now that pathing is handled
from sync import sync_user_data

# === CONFIG ===
SUMMONER_NAME = "RainbowThenga#420"
BASE_DATA_PATH = repo_root / "data" / "users"

# Optional filters — leave as None if not filtering
CHAMPION_NAME = None
START_TIME = None  # You could use something like int(datetime(2024, 6, 1).timestamp())
END_TIME = None


load_dotenv()
RIOT_API_KEY = os.getenv("RIOT_API_KEY")

if not RIOT_API_KEY:
    raise ValueError("RIOT_API_KEY not found in .env")

HEADERS = {
    "X-Riot-Token": RIOT_API_KEY
}

# Change these as needed
REGION_ROUTING = "europe"   # For match/v5 endpoints (matches & timelines)
PLATFORM_ROUTING = "euw1"      # For summoner-v4 endpoints (summoner info)

# Run the sync
sync_user_data(
    summoner_name=SUMMONER_NAME,
    base_data_path=BASE_DATA_PATH,
    champion_name=CHAMPION_NAME,
    start_time=START_TIME,
    end_time=END_TIME
)

RIOT_API_KEY: RGAPI-63d627c6-3446-46e8-8025-af5a7ea80c10

📥 Syncing data for summoner: RainbowThenga#420
   Champion filter: None
🔍 Found 0 new matches to check.
No new matches to download timelines for.

✅ Sync complete.



In [34]:
import json
from pathlib import Path
from collections import Counter
from api_handler import get_summoner_info
# Step 1: Get puuid
puuid = get_summoner_info(SUMMONER_NAME)["puuid"]

# Step 2: Define your data path where matches are saved
data_path = repo_root / "data" / "users" / SUMMONER_NAME.replace("#", "_") / "matches"

# Step 3: Load all match files
matches = []
for file in data_path.glob("*.json"):
    with open(file) as f:
        match = json.load(f)
        matches.append(match)

# Step 4: Extract champions you played in each match
champions = []
for match in matches:
    participants = match["info"]["participants"]
    player = next(p for p in participants if p["puuid"] == puuid)
    champions.append(player["championName"])

# Step 5: Count and display champions played
champion_counts = Counter(champions)
print("Champions played in synced matches:")
for champ, count in champion_counts.most_common():
    print(f"{champ}: {count}")

Champions played in synced matches:
Aphelios: 15
Kalista: 6
Jhin: 1
Kaisa: 1
